Environment

In [ ]:
libs = {
    "hypernet": "/home/zanardi/Codes/TC/HyperNet/hypernet",
    "pycomet": "/home/zanardi/Codes/ML/PyCOMET/pycomet"
}

In [ ]:
import sys
for lib in libs.values():
    sys.path.append(lib)

Import libraries

In [ ]:
import os
import json
import pycomet
import numpy as np
import tensorflow as tf

from scipy.linalg import expm
from silx.io.dictdump import h5todict

from hypernet import const
from hypernet.apps import utils as hyputils
from pycomet.networks import NodePartitionFunctions, EdgeBackward, LinearMEGraphNet, LinMEIntGraphNet

Plotting

In [ ]:
import matplotlib
from matplotlib import pyplot as plt
COLORS = matplotlib.rcParams['axes.prop_cycle'].by_key()['color']

path_to_figs = "./figs/"
os.makedirs(path_to_figs, exist_ok=True)

def plot_var(
    x,
    y,
    xref=None,
    yref=None,
    labels=[r'$t$ [s]', r'$X$'],
    scales=['log', 'log'],
    names=None,
    figname=None
):
    # Initialize figures
    fig = plt.figure()
    ax = fig.add_subplot()
    # x axis
    xlim = [np.amin(x), np.amax(x)]
    ax.set_xlim(xlim)
    ax.set_xlabel(labels[0])
    ax.set_xscale(scales[0])
    # y axis
    ax.set_ylabel(labels[1])
    ax.set_yscale(scales[1])
    # Plotting
    true_style = dict(
        lw=1
    )
    pred_style = dict(
        marker='^',
        fillstyle='none',
        markersize=5,
        markevery=y.shape[0]//20,
        ls='--',
        lw=1
    )
    for d in range(y.shape[1]):
        ax.plot(x, y[:,d], c=COLORS[d], **pred_style)
        ax.plot(xref, yref[:,d], c=COLORS[d], label=names[d], **true_style)
    plt.ticklabel_format(axis='y', style='sci', scilimits=(0,0))
    plt.legend()
    if (figname is not None):
        fig.savefig(figname)
    plt.show()
    plt.close()

Initializing thermophysical models

In [ ]:
with open("./../physics/inputs.json") as file:
    phyinp = json.load(file)
# Species thermo
sp_th = hyputils._get_specie_thermos(phyinp)
# Mixture
mix = hyputils._get_mixture(phyinp, sp_th)
# Kinetics
kin = hyputils._get_kinetics(phyinp, sp_th, mix)

Numerical solution

In [ ]:
sol = h5todict(h5file='./../physics/solution.hdf5')

In [ ]:
na, nm = [sol[f"X_{s}"] * sol['n'] for s in sp_th.keys()]

In [ ]:
# Compute source term
t, T = [sol[k] for k in ('t', 'T')]
n = np.hstack([na, nm])
omega = kin.function(t=t, y=[n, T])
dnmdt = omega[:,1:-1] / const.UNA
dnmdt = dnmdt.numpy().squeeze()

Analytical solution

In [ ]:
# Solution function
nm_int = lambda t, n0, k: np.matmul(expm(k*t), n0)

In [ ]:
# Construct the evolution matrix
kf, kb = 0.0, 0.0
for reaction in kin.reactions:
    kf = kf + reaction.fwd.k.numpy().squeeze()[0]
    kb = kb + reaction.bwd.k.numpy().squeeze()[0]
k = kb.T + kf
k = (k.T - np.diag(np.sum(k, axis=-1))) * na[0]
# Compute exact solution
nm_anal = np.vstack([nm_int(ti, nm[0], k) for ti in t])
# Compute source term
dnmdt_anal = np.einsum("ij,nj->ni",k,nm_anal) / const.UNA

In [ ]:
plot_var(
    x=t.squeeze()[1:],
    y=nm[1:],
    xref=t.squeeze()[1:],
    yref=nm_anal[1:],
    labels=[r'$t$ [s]', r'$n$ [m$^{-3}$]'],
    scales=['log', 'linear'],
    names=[r'O$_2^{(%d)}$' % (i+1) for i in range(dnmdt.shape[1])],
    figname=path_to_figs + "/solution"
)

Construct graph

In [ ]:
# # Nodes
# node_attr = nm[...,None]
# # Edges
# k = kf + kb.T
# k_idx = np.where(k > 0)
# edge_index = np.array(k_idx)
# edge_attr = (k[k_idx] * na[0]).reshape(1,-1,1)
# edge_attr = np.tile(edge_attr, (len(node_attr),1,1))

In [ ]:
# # Nodes
# nb_nodes = nm.shape[-1]
# node_t = np.log(np.tile(t.reshape(-1,1,1), (1,nb_nodes,1))+1e-15)
# node_n = nm[...,None]
# node_T = np.tile(T.reshape(-1,1,1), (1,nb_nodes,1))
# node_attr = np.concatenate([node_t, node_n, node_T], axis=-1)
# # Edges
# k_idx = np.where(kf > 0)
# edge_index = tf.convert_to_tensor(np.array(k_idx))
# edge_attr = (kf[k_idx] * na[0]).reshape(1,-1,1)
# edge_attr = np.tile(edge_attr, (len(node_attr),1,1))
# # Graph
# graph = [node_attr, edge_attr]
# graph = tf.nest.map_structure(tf.convert_to_tensor, graph)

In [ ]:
# Dimension
nb_pts, nb_nodes = nm.shape
# Nodes
node_n = nm[0].reshape(1,nb_nodes,1)
node_n = np.tile(node_n, (nb_pts,1,1))
node_T = T[0].reshape(1,1,1)
node_T = np.tile(node_T, (nb_pts,nb_nodes,1))
node_attr = np.concatenate([node_n, node_T], axis=-1)
# Edges
k_idx = np.where(kf > 0)
edge_attr = (kf[k_idx] * na[0]).reshape(1,-1,1)
edge_attr = np.tile(edge_attr, (nb_pts,1,1))
# Conds
cond_t = t.reshape(-1,1,1)
cond_t = np.tile(cond_t, (1,nb_nodes,1))
cond_n = na[0].reshape(1,1,1)
cond_n = np.tile(cond_n, (nb_pts,nb_nodes,1))
cond_attr = np.concatenate([cond_t, cond_n], axis=-1)
cond_attr = np.log(cond_attr + 1e-15)

In [ ]:
# Graph
graph = [node_attr, edge_attr, cond_attr]
graph = tf.nest.map_structure(tf.convert_to_tensor, graph)

Load linear master equation graph model

In [ ]:
tf.keras.backend.clear_session()

import network

net = LinMEIntGraphNet(dim=network.dim, lays_propty=network.lays_propty)

In [ ]:
outputs = net.call(net.inputs)

In [ ]:
model = tf.keras.Model(net.inputs, outputs)

In [ ]:
model.summary()

In [ ]:
y = net.call(graph)

In [ ]:
y

In [ ]:
yt = tf.gather(graph[0], [1], axis=-1)
with tf.GradientTape() as g:
  g.watch(graph[0])
  y = model(graph) / const.UNA
  e = tf.reduce_mean(tf.abs(y-yt), axis=[1,2], keepdims=1)
dy_dx = g.batch_jacobian(e, graph[0])
(dy_dx.numpy() != 0.0).any()

In [ ]:
dnmdt_pred = model(graph) / const.UNA
dnmdt_pred = dnmdt_pred.numpy().squeeze()

In [ ]:
plot_var(
    x=t.squeeze()[1:],
    y=dnmdt_pred[1:],
    xref=t.squeeze()[1:],
    yref=dnmdt[1:],
    labels=[r'$t$ [s]', r'$\omega$ [m$^{-3}$s$^{-1}$]'],
    scales=['log', 'linear'],
    names=[r'O$_2^{(%d)}$' % (i+1) for i in range(dnmdt.shape[1])],
    figname=path_to_figs + "/source"
)